# NASA POWER API Test — CropSage

## Objectives

1. Verify that NASA POWER works without authentication.
2. Test the Climatology Point endpoint.
3. Test the Daily Point endpoint.
4. Confirm agricultural variables, definitions and units.
5. Detect sentinel, null and absent values.
6. Test additional crop-relevant variables:
   - wind speed;
   - dew point;
   - photosynthetically active radiation;
   - root-zone soil wetness;
   - evapotranspiration.
7. Verify API validation-error behaviour.
8. Decide which fields CropSage will use in production.
9. Document how NASA POWER complements FortyGuard.

In [1]:
import json
from pathlib import Path

import pandas as pd
import requests

print("Imports successful")

Imports successful


In [2]:
FARM_ID = "plainview_demo"
FARM_NAME = "Plainview demonstration farm"

LATITUDE = 34.1800
LONGITUDE = -101.7600
TIMEZONE = "America/Chicago"

FULL_YEAR_START = "20250101"
FULL_YEAR_END = "20251231"


PARAMETERS = [
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "PRECTOTCORR",
    "RH2M",
    "ALLSKY_SFC_SW_DWN",
]

COMMUNITY = "AG"
TIMEOUT_SECONDS = 60

In [3]:
PARAMETER_PURPOSES = {
    "T2M": (
        "Climatological mean air temperature "
        "at 2 metres"
    ),
    "T2M_MAX": (
        "Historical extreme maximum air "
        "temperature at 2 metres"
    ),
    "T2M_MIN": (
        "Historical extreme minimum air "
        "temperature at 2 metres"
    ),
    "PRECTOTCORR": (
        "Corrected average precipitation rate"
    ),
    "RH2M": (
        "Relative humidity at 2 metres"
    ),
    "ALLSKY_SFC_SW_DWN": (
        "All-sky surface shortwave radiation"
    ),
}

pd.Series(
    PARAMETER_PURPOSES,
    name="Purpose",
)

T2M                    Climatological mean air temperature at 2 metres
T2M_MAX              Historical extreme maximum air temperature at ...
T2M_MIN              Historical extreme minimum air temperature at ...
PRECTOTCORR                       Corrected average precipitation rate
RH2M                                     Relative humidity at 2 metres
ALLSKY_SFC_SW_DWN                  All-sky surface shortwave radiation
Name: Purpose, dtype: str

In [4]:
CLIMATOLOGY_URL = (
    "https://power.larc.nasa.gov/api/temporal/climatology/point"
)

climatology_params = {
    "parameters": ",".join(PARAMETERS),
    "community": COMMUNITY,
    "longitude": LONGITUDE,
    "latitude": LATITUDE,
    "format": "JSON",
}

climatology_response = requests.get(
    CLIMATOLOGY_URL,
    params=climatology_params,
    timeout=TIMEOUT_SECONDS,
)

print("Request URL:")
print(climatology_response.url)
print()
print("HTTP status:", climatology_response.status_code)
print("Content-Type:", climatology_response.headers.get("content-type"))

climatology_response.raise_for_status()
climatology_data = climatology_response.json()

Request URL:
https://power.larc.nasa.gov/api/temporal/climatology/point?parameters=T2M%2CT2M_MAX%2CT2M_MIN%2CPRECTOTCORR%2CRH2M%2CALLSKY_SFC_SW_DWN&community=AG&longitude=-101.76&latitude=34.18&format=JSON

HTTP status: 200
Content-Type: application/json


In [5]:
print("Top-level fields:")
print(list(climatology_data.keys()))

print("\nGeometry:")
print(json.dumps(climatology_data.get("geometry"), indent=2))

print("\nMessages:")
print(climatology_data.get("messages"))

Top-level fields:
['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times']

Geometry:
{
  "type": "Point",
  "coordinates": [
    -101.76,
    34.18,
    1034.08
  ]
}

Messages:
['The requested parameters are retrieved from a pre-computed climatological period (January 2001 - December 2020)']


In [6]:
parameter_values = climatology_data["properties"]["parameter"]

returned_parameters = list(parameter_values.keys())

print("Requested:", PARAMETERS)
print("Returned:", returned_parameters)

missing_parameters = sorted(set(PARAMETERS) - set(returned_parameters))
unexpected_parameters = sorted(set(returned_parameters) - set(PARAMETERS))

print("Missing:", missing_parameters)
print("Unexpected:", unexpected_parameters)

Requested: ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'ALLSKY_SFC_SW_DWN']
Returned: ['ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN']
Missing: []
Unexpected: []


In [7]:
parameter_metadata = climatology_data.get("parameters", {})

metadata_rows = []

for code in returned_parameters:
    metadata = parameter_metadata.get(code, {})

    metadata_rows.append({
        "code": code,
        "purpose": PARAMETER_PURPOSES.get(code),
        "long_name": metadata.get("longname"),
        "units": metadata.get("units"),
    })

metadata_df = pd.DataFrame(metadata_rows)
metadata_df

,code,purpose,long_name,units
0,ALLSKY_SFC_SW_DWN,All-sky surface shortwave radiation,All Sky Surface Shortwave Downward Irradiance,MJ/m^2/day
1,PRECTOTCORR,Corrected average precipitation rate,Precipitation Corrected,mm/day
2,RH2M,Relative humidity at 2 metres,Relative Humidity at 2 Meters,%
3,T2M,Climatological mean air temperature at 2 metres,Temperature at 2 Meters,C
4,T2M_MAX,Historical extreme maximum air temperature at ...,Temperature at 2 Meters Maximum,C
5,T2M_MIN,Historical extreme minimum air temperature at ...,Temperature at 2 Meters Minimum,C


In [9]:
climatology_df = pd.DataFrame(parameter_values)

month_order = [
    "JAN", "FEB", "MAR", "APR", "MAY", "JUN",
    "JUL", "AUG", "SEP", "OCT", "NOV", "DEC", "ANN"
]

existing_rows = [
    month for month in month_order
    if month in climatology_df.index
]

climatology_df = climatology_df.loc[existing_rows]
climatology_df

,ALLSKY_SFC_SW_DWN,PRECTOTCORR,RH2M,T2M,T2M_MAX,T2M_MIN
JAN,11.72,0.52,56.06,3.96,25.49,-13.20
FEB,14.46,0.59,53.99,5.92,29.71,-15.23
MAR,18.95,1.07,48.98,11.27,32.94,-9.74
APR,23.24,1.01,43.52,16.11,39.07,-4.61
MAY,25.29,2.02,46.50,21.18,40.26,-0.14
JUN,27.22,2.21,49.46,26.23,44.69,9.41
JUL,26.13,1.82,49.32,27.66,44.33,14.13
AUG,23.54,1.87,51.29,26.96,42.59,12.46
SEP,19.72,2.09,56.24,22.40,39.07,6.11
OCT,15.65,1.48,55.20,15.81,36.47,-6.40


In [10]:
MISSING_SENTINELS = [-999, -999.0, -9999, -9999.0]

missing_report = pd.DataFrame({
    "sentinel_count": {
        column: climatology_df[column]
        .isin(MISSING_SENTINELS)
        .sum()
        for column in climatology_df.columns
    },
    "null_count": climatology_df.isna().sum(),
})

missing_report

,sentinel_count,null_count
ALLSKY_SFC_SW_DWN,0,0
PRECTOTCORR,0,0
RH2M,0,0
T2M,0,0
T2M_MAX,0,0
T2M_MIN,0,0


In [11]:
DAYS_PER_MONTH = {
    "JAN": 31,
    "FEB": 28.25,
    "MAR": 31,
    "APR": 30,
    "MAY": 31,
    "JUN": 30,
    "JUL": 31,
    "AUG": 31,
    "SEP": 30,
    "OCT": 31,
    "NOV": 30,
    "DEC": 31,
}

monthly_climatology_df = climatology_df.drop(
    index="ANN",
    errors="ignore",
).copy()

monthly_climatology_df["PRECTOTCORR_MONTHLY_MM"] = [
    monthly_climatology_df.loc[month, "PRECTOTCORR"]
    * DAYS_PER_MONTH[month]
    for month in monthly_climatology_df.index
]

monthly_climatology_df[
    ["PRECTOTCORR", "PRECTOTCORR_MONTHLY_MM"]
]

,PRECTOTCORR,PRECTOTCORR_MONTHLY_MM
JAN,0.52,16.1200
FEB,0.59,16.6675
MAR,1.07,33.1700
APR,1.01,30.3000
MAY,2.02,62.6200
JUN,2.21,66.3000
JUL,1.82,56.4200
AUG,1.87,57.9700
SEP,2.09,62.7000
OCT,1.48,45.8800


In [12]:
approximate_annual_precipitation_mm = (
    monthly_climatology_df[
        "PRECTOTCORR_MONTHLY_MM"
    ].sum()
)

print(
    "Approximate annual precipitation:",
    round(approximate_annual_precipitation_mm, 2),
    "mm",
)

Approximate annual precipitation: 485.86 mm


In [13]:
DAILY_URL = (
    "https://power.larc.nasa.gov/api/temporal/daily/point"
)

DAILY_PARAMETERS = [
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "PRECTOTCORR",
    "RH2M",
    "ALLSKY_SFC_SW_DWN",
]

daily_params = {
    "parameters": ",".join(DAILY_PARAMETERS),
    "community": COMMUNITY,
    "longitude": LONGITUDE,
    "latitude": LATITUDE,
    "start": "20250701",
    "end": "20250707",
    "format": "JSON",
    "time-standard": "LST",
}

daily_response = requests.get(
    DAILY_URL,
    params=daily_params,
    timeout=TIMEOUT_SECONDS,
)

print("HTTP status:", daily_response.status_code)
print("Request URL:")
print(daily_response.url)

daily_response.raise_for_status()
daily_data = daily_response.json()

HTTP status: 200
Request URL:
https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M%2CT2M_MAX%2CT2M_MIN%2CPRECTOTCORR%2CRH2M%2CALLSKY_SFC_SW_DWN&community=AG&longitude=-101.76&latitude=34.18&start=20250701&end=20250707&format=JSON&time-standard=LST


In [14]:
daily_values = daily_data["properties"]["parameter"]

daily_df = pd.DataFrame(daily_values)

daily_df.index = pd.to_datetime(
    daily_df.index,
    format="%Y%m%d",
)

daily_df.index.name = "date"

daily_df

,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,ALLSKY_SFC_SW_DWN
date,,,,,,
2025-07-01,21.03,23.70,18.28,3.42,81.83,11.45
2025-07-02,22.88,29.34,19.08,3.51,73.70,15.13
2025-07-03,22.80,26.86,19.02,17.24,83.87,10.33
2025-07-04,26.29,33.44,20.35,0.51,69.66,23.90
2025-07-05,26.67,33.62,20.72,1.60,70.20,19.27
2025-07-06,26.56,32.76,19.86,0.85,65.97,28.19
2025-07-07,26.00,31.48,20.81,4.82,69.35,28.22


In [15]:
daily_missing_report = pd.DataFrame({
    "sentinel_count": {
        column: daily_df[column]
        .isin(MISSING_SENTINELS)
        .sum()
        for column in daily_df.columns
    },
    "null_count": daily_df.isna().sum(),
})

daily_missing_report

,sentinel_count,null_count
T2M,0,0
T2M_MAX,0,0
T2M_MIN,0,0
PRECTOTCORR,0,0
RH2M,0,0
ALLSKY_SFC_SW_DWN,0,0


In [16]:
invalid_params = {
    "parameters": "NOT_A_REAL_PARAMETER",
    "community": COMMUNITY,
    "longitude": LONGITUDE,
    "latitude": LATITUDE,
    "format": "JSON",
}

invalid_response = requests.get(
    CLIMATOLOGY_URL,
    params=invalid_params,
    timeout=TIMEOUT_SECONDS,
)

print("HTTP status:", invalid_response.status_code)

try:
    invalid_error_data = invalid_response.json()
    print(json.dumps(invalid_error_data, indent=2))
except ValueError:
    print(invalid_response.text[:2000])

HTTP status: 422
{
  "header": "The POWER Climatology API failed to complete your request; please review the errors below and the POWER Docs (https://power.larc.nasa.gov/docs/).",
  "messages": [
    "One of your parameters is incorrect: NOT_A_REAL_PARAMETER."
  ]
}


In [17]:
ADDITIONAL_NASA_PARAMETERS = [
    "WS2M",                 # Wind speed at 2 m
    "T2MDEW",              # Dew/frost point at 2 m
    "ALLSKY_SFC_PAR_TOT",  # Photosynthetically active radiation
    "GWETROOT",             # Root-zone soil wetness index
    "EVPTRNS",              # Evapotranspiration
]

ADDITIONAL_NASA_PARAMETERS

['WS2M', 'T2MDEW', 'ALLSKY_SFC_PAR_TOT', 'GWETROOT', 'EVPTRNS']

In [18]:
ADDITIONAL_PARAMETER_PURPOSES = {
    "WS2M": "Wind speed at 2 metres",
    "T2MDEW": "Dew or frost-point temperature at 2 metres",
    "ALLSKY_SFC_PAR_TOT": "Photosynthetically active radiation",
    "GWETROOT": "Root-zone soil wetness index",
    "EVPTRNS": "Evapotranspiration",
}

additional_climatology_params = {
    "parameters": ",".join(ADDITIONAL_NASA_PARAMETERS),
    "community": COMMUNITY,
    "longitude": LONGITUDE,
    "latitude": LATITUDE,
    "format": "JSON",
}

additional_climatology_response = requests.get(
    CLIMATOLOGY_URL,
    params=additional_climatology_params,
    timeout=TIMEOUT_SECONDS,
)

print("HTTP status:", additional_climatology_response.status_code)
print("Request URL:")
print(additional_climatology_response.url)

if not additional_climatology_response.ok:
    print(additional_climatology_response.text)

additional_climatology_response.raise_for_status()
additional_climatology_data = additional_climatology_response.json()

HTTP status: 200
Request URL:
https://power.larc.nasa.gov/api/temporal/climatology/point?parameters=WS2M%2CT2MDEW%2CALLSKY_SFC_PAR_TOT%2CGWETROOT%2CEVPTRNS&community=AG&longitude=-101.76&latitude=34.18&format=JSON


In [19]:
additional_climatology_values = (
    additional_climatology_data["properties"]["parameter"]
)

additional_returned_parameters = list(
    additional_climatology_values.keys()
)

additional_missing_parameters = sorted(
    set(ADDITIONAL_NASA_PARAMETERS)
    - set(additional_returned_parameters)
)

additional_unexpected_parameters = sorted(
    set(additional_returned_parameters)
    - set(ADDITIONAL_NASA_PARAMETERS)
)

print("Requested:", ADDITIONAL_NASA_PARAMETERS)
print("Returned:", additional_returned_parameters)
print("Missing:", additional_missing_parameters)
print("Unexpected:", additional_unexpected_parameters)

additional_metadata = additional_climatology_data.get(
    "parameters", {}
)

additional_metadata_rows = []

for code in additional_returned_parameters:
    metadata = additional_metadata.get(code, {})

    additional_metadata_rows.append({
        "code": code,
        "purpose": ADDITIONAL_PARAMETER_PURPOSES.get(code),
        "long_name": metadata.get("longname"),
        "units": metadata.get("units"),
    })

additional_metadata_df = pd.DataFrame(
    additional_metadata_rows
)

print("\nMetadata:")
print(additional_metadata_df.to_string(index=False))

additional_climatology_df = pd.DataFrame(
    additional_climatology_values
)

additional_climatology_df.index.name = "period"
additional_climatology_df

Requested: ['WS2M', 'T2MDEW', 'ALLSKY_SFC_PAR_TOT', 'GWETROOT', 'EVPTRNS']
Returned: ['ALLSKY_SFC_PAR_TOT', 'EVPTRNS', 'GWETROOT', 'T2MDEW', 'WS2M']
Missing: []
Unexpected: []

Metadata:
              code                                    purpose                      long_name      units
ALLSKY_SFC_PAR_TOT        Photosynthetically active radiation      All Sky Surface Total PAR MJ/m^2/day
           EVPTRNS                         Evapotranspiration Evapotranspiration Energy Flux MJ/m^2/day
          GWETROOT               Root-zone soil wetness index         Root Zone Soil Wetness          1
            T2MDEW Dew or frost-point temperature at 2 metres    Dew/Frost Point at 2 Meters          C
              WS2M                     Wind speed at 2 metres         Wind Speed at 2 Meters        m/s


,ALLSKY_SFC_PAR_TOT,EVPTRNS,GWETROOT,T2MDEW,WS2M
period,,,,,
JAN,5.15,0.06,0.39,-5.08,3.34
FEB,6.40,0.14,0.39,-4.09,3.63
MAR,8.45,0.34,0.38,-1.02,3.80
APR,10.42,0.56,0.38,1.25,4.17
MAY,11.49,0.97,0.38,6.44,3.81
JUN,12.56,1.49,0.37,12.81,3.70
JUL,12.20,1.31,0.37,14.55,3.12
AUG,10.99,1.15,0.36,14.58,2.90
SEP,9.12,1.07,0.38,11.83,2.97


In [20]:
additional_climatology_missing_report = pd.DataFrame({
    "sentinel_count": {
        column: int(
            additional_climatology_df[column]
            .isin(MISSING_SENTINELS)
            .sum()
        )
        for column in additional_climatology_df.columns
    },
    "null_count": {
        column: int(
            additional_climatology_df[column]
            .isna()
            .sum()
        )
        for column in additional_climatology_df.columns
    },
})

additional_climatology_missing_report

,sentinel_count,null_count
ALLSKY_SFC_PAR_TOT,0,0
EVPTRNS,0,0
GWETROOT,0,0
T2MDEW,0,0
WS2M,0,0


In [21]:
additional_daily_params = {
    "parameters": ",".join(ADDITIONAL_NASA_PARAMETERS),
    "community": COMMUNITY,
    "longitude": LONGITUDE,
    "latitude": LATITUDE,
    "start": daily_params["start"],
    "end": daily_params["end"],
    "format": "JSON",
    "time-standard": "LST",
}

additional_daily_response = requests.get(
    DAILY_URL,
    params=additional_daily_params,
    timeout=TIMEOUT_SECONDS,
)

print("HTTP status:", additional_daily_response.status_code)
print("Request URL:")
print(additional_daily_response.url)

if not additional_daily_response.ok:
    print(additional_daily_response.text)

additional_daily_response.raise_for_status()
additional_daily_data = additional_daily_response.json()

HTTP status: 200
Request URL:
https://power.larc.nasa.gov/api/temporal/daily/point?parameters=WS2M%2CT2MDEW%2CALLSKY_SFC_PAR_TOT%2CGWETROOT%2CEVPTRNS&community=AG&longitude=-101.76&latitude=34.18&start=20250701&end=20250707&format=JSON&time-standard=LST


In [22]:
additional_daily_values = (
    additional_daily_data["properties"]["parameter"]
)

additional_daily_returned = list(
    additional_daily_values.keys()
)

additional_daily_missing = sorted(
    set(ADDITIONAL_NASA_PARAMETERS)
    - set(additional_daily_returned)
)

print("Returned:", additional_daily_returned)
print("Missing:", additional_daily_missing)

additional_daily_df = pd.DataFrame(
    additional_daily_values
)

additional_daily_df.index = pd.to_datetime(
    additional_daily_df.index,
    format="%Y%m%d",
)

additional_daily_df.index.name = "date"

additional_daily_missing_report = pd.DataFrame({
    "sentinel_count": {
        column: int(
            additional_daily_df[column]
            .isin(MISSING_SENTINELS)
            .sum()
        )
        for column in additional_daily_df.columns
    },
    "null_count": {
        column: int(
            additional_daily_df[column]
            .isna()
            .sum()
        )
        for column in additional_daily_df.columns
    },
})

print("\nMissing-value report:")
print(additional_daily_missing_report)

additional_daily_df

Returned: ['WS2M', 'T2MDEW', 'ALLSKY_SFC_PAR_TOT', 'GWETROOT', 'EVPTRNS']
Missing: []

Missing-value report:
                    sentinel_count  null_count
WS2M                             0           0
T2MDEW                           0           0
ALLSKY_SFC_PAR_TOT               0           0
GWETROOT                         0           0
EVPTRNS                          0           0


,WS2M,T2MDEW,ALLSKY_SFC_PAR_TOT,GWETROOT,EVPTRNS
date,,,,,
2025-07-01,2.68,17.66,5.91,0.39,0.45
2025-07-02,3.14,17.34,7.54,0.39,1.57
2025-07-03,3.38,19.76,5.40,0.41,1.22
2025-07-04,2.90,19.27,11.31,0.42,5.59
2025-07-05,2.82,20.01,9.29,0.41,5.14
2025-07-06,1.41,18.72,13.06,0.40,4.42
2025-07-07,1.51,19.32,13.13,0.40,3.03


## One year test 2025


In [23]:
ALL_NASA_PARAMETERS = list(dict.fromkeys(
    PARAMETERS + ADDITIONAL_NASA_PARAMETERS
))

FULL_YEAR_START = "20250101"
FULL_YEAR_END = "20251231"

full_year_params = {
    "parameters": ",".join(ALL_NASA_PARAMETERS),
    "community": COMMUNITY,
    "longitude": LONGITUDE,
    "latitude": LATITUDE,
    "start": FULL_YEAR_START,
    "end": FULL_YEAR_END,
    "format": "JSON",
    "time-standard": "LST",
}

full_year_response = requests.get(
    DAILY_URL,
    params=full_year_params,
    timeout=TIMEOUT_SECONDS,
)

print("HTTP status:", full_year_response.status_code)
print("Request URL:")
print(full_year_response.url)

if not full_year_response.ok:
    print(full_year_response.text)

full_year_response.raise_for_status()
full_year_data = full_year_response.json()

HTTP status: 200
Request URL:
https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M%2CT2M_MAX%2CT2M_MIN%2CPRECTOTCORR%2CRH2M%2CALLSKY_SFC_SW_DWN%2CWS2M%2CT2MDEW%2CALLSKY_SFC_PAR_TOT%2CGWETROOT%2CEVPTRNS&community=AG&longitude=-101.76&latitude=34.18&start=20250101&end=20251231&format=JSON&time-standard=LST


In [24]:
full_year_values = full_year_data["properties"]["parameter"]
full_year_returned = list(full_year_values.keys())

missing_parameters = sorted(
    set(ALL_NASA_PARAMETERS) - set(full_year_returned)
)

unexpected_parameters = sorted(
    set(full_year_returned) - set(ALL_NASA_PARAMETERS)
)

raw_full_year_df = pd.DataFrame(full_year_values)

raw_full_year_df.index = pd.to_datetime(
    raw_full_year_df.index,
    format="%Y%m%d",
)

raw_full_year_df.index.name = "date"
raw_full_year_df = raw_full_year_df.sort_index()

quality_report = pd.DataFrame({
    "sentinel_count": {
        column: int(
            raw_full_year_df[column]
            .isin(MISSING_SENTINELS)
            .sum()
        )
        for column in raw_full_year_df.columns
    },
    "null_count": {
        column: int(
            raw_full_year_df[column].isna().sum()
        )
        for column in raw_full_year_df.columns
    },
})

full_year_df = raw_full_year_df.mask(
    raw_full_year_df.isin(MISSING_SENTINELS)
)

expected_dates = pd.date_range(
    FULL_YEAR_START,
    FULL_YEAR_END,
    freq="D",
)

missing_dates = expected_dates.difference(
    full_year_df.index
)

print("Requested parameters:", len(ALL_NASA_PARAMETERS))
print("Returned parameters:", len(full_year_returned))
print("Missing parameters:", missing_parameters)
print("Unexpected parameters:", unexpected_parameters)
print("Returned days:", len(full_year_df))
print("Missing dates:", len(missing_dates))

quality_report

Requested parameters: 11
Returned parameters: 11
Missing parameters: []
Unexpected parameters: []
Returned days: 365
Missing dates: 0


,sentinel_count,null_count
T2M,0,0
T2M_MAX,0,0
T2M_MIN,0,0
PRECTOTCORR,0,0
RH2M,0,0
ALLSKY_SFC_SW_DWN,0,0
WS2M,0,0
T2MDEW,0,0
ALLSKY_SFC_PAR_TOT,0,0
GWETROOT,0,0


In [25]:
import numpy as np

# Approximate daily vapour pressure deficit.
saturation_vapour_pressure = (
    0.6108
    * np.exp(
        (17.27 * full_year_df["T2M"])
        / (full_year_df["T2M"] + 237.3)
    )
)

actual_vapour_pressure = (
    0.6108
    * np.exp(
        (17.27 * full_year_df["T2MDEW"])
        / (full_year_df["T2MDEW"] + 237.3)
    )
)

full_year_df["VPD_KPA"] = (
    saturation_vapour_pressure
    - actual_vapour_pressure
).clip(lower=0)

# Daily temperature range.
full_year_df["TEMPERATURE_RANGE_C"] = (
    full_year_df["T2M_MAX"]
    - full_year_df["T2M_MIN"]
)

# Demonstration values only.
# Production values will be calculated separately for every crop.
EXAMPLE_BASE_TEMPERATURE_C = 10
EXAMPLE_HEAT_THRESHOLD_C = 35

full_year_df["GDD_BASE_10"] = (
    (
        full_year_df["T2M_MAX"]
        + full_year_df["T2M_MIN"]
    ) / 2
    - EXAMPLE_BASE_TEMPERATURE_C
).clip(lower=0)

full_year_df["HEAT_STRESS_DAY_35C"] = (
    full_year_df["T2M_MAX"]
    >= EXAMPLE_HEAT_THRESHOLD_C
).astype(int)

full_year_df.head()

,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,ALLSKY_SFC_SW_DWN,WS2M,T2MDEW,ALLSKY_SFC_PAR_TOT,GWETROOT,EVPTRNS,VPD_KPA,TEMPERATURE_RANGE_C,GDD_BASE_10,HEAT_STRESS_DAY_35C
date,,,,,,,,,,,,,,,
2025-01-01,1.20,7.28,-4.15,0.0,76.95,7.24,2.94,-2.83,3.30,0.38,0.01,0.170373,11.43,0.0,0
2025-01-02,6.79,19.07,-0.73,0.0,61.48,12.01,2.86,-1.96,5.23,0.38,0.14,0.458530,19.80,0.0,0
2025-01-03,5.25,14.35,-2.09,0.0,72.57,12.40,2.94,0.09,5.37,0.38,0.06,0.272839,16.44,0.0,0
2025-01-04,8.28,16.07,3.16,0.0,62.40,8.07,3.67,0.45,3.66,0.38,0.04,0.462304,12.91,0.0,0
2025-01-05,-0.36,6.51,-6.66,0.0,51.38,13.17,5.43,-9.24,5.65,0.38,0.00,0.291578,13.17,0.0,0


In [26]:
monthly_features = full_year_df.resample("MS").agg({
    "T2M": "mean",
    "T2M_MAX": "max",
    "T2M_MIN": "min",
    "PRECTOTCORR": "sum",
    "RH2M": "mean",
    "ALLSKY_SFC_SW_DWN": "mean",
    "WS2M": "mean",
    "T2MDEW": "mean",
    "ALLSKY_SFC_PAR_TOT": "mean",
    "GWETROOT": "mean",
    "EVPTRNS": "sum",
    "VPD_KPA": "mean",
    "GDD_BASE_10": "sum",
    "HEAT_STRESS_DAY_35C": "sum",
})

monthly_features.index = monthly_features.index.strftime("%b")

monthly_features = monthly_features.rename(columns={
    "PRECTOTCORR": "PRECIPITATION_TOTAL_MM",
    "EVPTRNS": "EVPTRNS_TOTAL_MJ_M2",
})

annual_summary = pd.Series({
    "mean_temperature_c": full_year_df["T2M"].mean(),
    "highest_temperature_c": full_year_df["T2M_MAX"].max(),
    "lowest_temperature_c": full_year_df["T2M_MIN"].min(),
    "annual_precipitation_mm": full_year_df["PRECTOTCORR"].sum(),
    "mean_relative_humidity_percent": full_year_df["RH2M"].mean(),
    "mean_wind_speed_m_s": full_year_df["WS2M"].mean(),
    "mean_solar_radiation_mj_m2_day":
        full_year_df["ALLSKY_SFC_SW_DWN"].mean(),
    "mean_par_mj_m2_day":
        full_year_df["ALLSKY_SFC_PAR_TOT"].mean(),
    "mean_root_zone_wetness":
        full_year_df["GWETROOT"].mean(),
    "mean_vpd_kpa": full_year_df["VPD_KPA"].mean(),
    "gdd_base_10": full_year_df["GDD_BASE_10"].sum(),
    "heat_stress_days_above_35c":
        full_year_df["HEAT_STRESS_DAY_35C"].sum(),
}).round(2)

print("Annual summary:")
print(annual_summary)

monthly_features.round(2)

Annual summary:
mean_temperature_c                  17.06
highest_temperature_c               40.95
lowest_temperature_c               -12.05
annual_precipitation_mm            432.61
mean_relative_humidity_percent      50.33
mean_wind_speed_m_s                  3.39
mean_solar_radiation_mj_m2_day      18.76
mean_par_mj_m2_day                   8.51
mean_root_zone_wetness               0.37
mean_vpd_kpa                         1.19
gdd_base_10                       3279.23
heat_stress_days_above_35c          44.00
dtype: float64


,T2M,T2M_MAX,T2M_MIN,PRECIPITATION_TOTAL_MM,RH2M,ALLSKY_SFC_SW_DWN,WS2M,T2MDEW,ALLSKY_SFC_PAR_TOT,GWETROOT,EVPTRNS_TOTAL_MJ_M2,VPD_KPA,GDD_BASE_10,HEAT_STRESS_DAY_35C
date,,,,,,,,,,,,,,
Jan,1.30,19.07,-12.05,5.37,57.90,11.95,3.35,-6.84,5.18,0.39,1.10,0.31,0.00,0
Feb,6.12,29.73,-11.54,0.42,49.20,14.42,3.37,-5.08,6.34,0.37,3.62,0.60,49.70,0
Mar,13.46,31.30,-0.77,6.68,37.15,20.00,4.36,-2.89,8.81,0.36,3.91,1.06,136.00,0
Apr,17.58,34.80,-1.47,72.58,47.98,20.99,4.32,4.08,9.48,0.37,17.19,1.20,245.37,0
May,20.75,38.15,6.14,72.43,47.90,25.37,3.53,6.82,11.43,0.39,46.47,1.49,339.86,4
Jun,26.05,40.95,12.20,100.07,58.42,25.54,3.55,16.09,11.93,0.39,64.53,1.56,487.56,8
Jul,27.12,38.98,18.28,73.95,57.45,24.80,2.90,16.76,11.61,0.38,71.26,1.70,535.04,14
Aug,28.02,40.39,18.78,55.81,49.81,22.66,3.02,15.02,10.54,0.36,20.31,2.08,570.67,16
Sep,23.73,36.85,12.86,25.01,52.08,19.59,2.83,12.10,9.05,0.37,28.73,1.52,420.88,2


In [28]:
.data/evidence/providers/nasa_power_plainview.json

SyntaxError: invalid syntax (974761473.py, line 1)

# Final NASA POWER Integration Decision

## How CropSage Will Use NASA POWER

CropSage will use two NASA POWER endpoints for different purposes.

---

## 1. Climatology Point Endpoint

**Endpoint**

`GET /api/temporal/climatology/point`

**Purpose**

This is the primary NASA POWER endpoint used by CropSage. It provides a compact, long-term regional climate baseline for the farmer's location.

The tested response represents the precomputed climatology period from January 2001 through December 2020.

**Variables used**

- `T2M`, `T2M_MAX`, `T2M_MIN`: mean temperature and historical extreme-temperature context
- `PRECTOTCORR`: average precipitation in mm/day
- `RH2M`: relative humidity in %
- `ALLSKY_SFC_SW_DWN`: all-sky surface solar radiation in MJ/m²/day
- `WS2M`: wind speed at 2 metres in m/s
- `T2MDEW`: dew/frost-point temperature at 2 metres in °C
- `ALLSKY_SFC_PAR_TOT`: photosynthetically active radiation in MJ/m²/day
- `GWETROOT`: regional root-zone soil-wetness index
- `EVPTRNS`: modeled evapotranspiration energy flux in MJ/m²/day

**Derived values**

CropSage will calculate:

- estimated monthly precipitation totals;
- average temperature during the proposed growing months;
- total growing-season precipitation;
- average growing-season humidity, wind speed, solar radiation and PAR;
- approximate vapour-pressure deficit from mean temperature and dew point;
- crop-specific growing degree days and heat-stress days;
- regional extreme-heat, extreme-cold and atmospheric-dryness warnings.

**How it affects recommendations**

The climatology data will be compared with each crop's verified requirements to calculate the long-term regional climate-fit component.

The extreme maximum and minimum values will be shown as risk context. They will not be treated as normal daily conditions. `GWETROOT` and `EVPTRNS` will remain contextual rather than decisive scoring inputs until they are validated against local soil and irrigation evidence.

---

## 2. Daily Point Endpoint

**Endpoint**

`GET /api/temporal/daily/point`

**Purpose**

This endpoint provides historical values for individual dates. CropSage will use bounded, cached daily history to create a detailed growing-season profile for the selected demonstration farm. A complete 2025 request was successfully tested.

**Variables used**

The daily request uses all 11 tested parameters: temperature, precipitation, relative humidity, solar radiation, wind speed, dew point, PAR, regional root-zone wetness and evapotranspiration energy flux.

**Potential derived indicators**

- crop-specific growing degree days;
- historical days above a crop-specific heat threshold;
- historical frost days;
- number of rainy days;
- dry-spell duration;
- historical daily temperature range;
- approximate vapour-pressure deficit;
- monthly wind, solar-radiation and PAR summaries.

**MVP limitation**

The application will not download decades of daily data during every user request.

For the hackathon MVP, one bounded year of daily history will be precomputed and cached for the selected demonstration farm. Additional locations will use bounded requests and cache the first successful response.

The Daily endpoint is supporting evidence. It does not replace FortyGuard's local heat intelligence.

---

## Relationship With FortyGuard

NASA POWER and FortyGuard have separate responsibilities.

| Source | CropSage responsibility |
|---|---|
| NASA climatology | Long-term regional temperature, water, humidity, wind and solar baseline |
| NASA daily history | Historical growing-season behaviour and derived VPD, GDD and stress-day features |
| FortyGuard heatmap | Local temperature distribution |
| FortyGuard exceedance | Hours above a crop-specific threshold |
| FortyGuard persistence | Longest continuous heat-stress period |
| FortyGuard forecast | Immediate heat warning up to the supported forecast horizon |

NASA POWER is coarse regional modeled data. It must not be presented as field-level measurement.

FortyGuard remains technically central because it supplies the local heat evidence that can directly change a crop's risk status or ranking.

---

## Recommendation Workflow

1. The farmer provides a location and planting period.
2. CropSage retrieves or loads the NASA monthly climatology.
3. CropSage selects the months corresponding to each crop's proposed growing period.
4. CropSage loads bounded, cached NASA daily history and calculates crop-specific VPD, GDD, heat, frost, rain and dry-spell indicators.
5. CropSage retrieves local heat evidence from FortyGuard.
6. The deterministic engine compares the normalized evidence with the verified crop catalog.
7. CropSage calculates climate fit, heat risk, soil fit, water fit and confidence.
8. The agent explains the validated top-three crop recommendations and their limitations.

---

## Important Interpretation Limits

NASA POWER does not provide an exact future seasonal forecast.

CropSage will not claim that NASA data predicts the daily temperature or rainfall several months ahead. NASA POWER is coarse regional modeled data rather than a field sensor or field-resolution forecast.

`GWETROOT` must not be presented as measured farm soil moisture. `EVPTRNS` will not be used directly as crop irrigation demand because the tested summer values were near zero and require local validation.

Recommendations represent:

- long-term climate suitability;
- historical growing-season behaviour;
- current or recent local heat evidence;
- short-range heat alerts;
- clearly labelled hotter, drier or irrigation scenarios.

The result is preliminary crop-planning decision support, not a guarantee of yield or profit.

---

## Test Outcome

- Authentication required: No
- Climatology endpoint: Successful
- Daily endpoint: Successful
- Parameters tested: 11 requested and 11 returned
- Full-year daily test: 365 days returned for 2025
- Missing dates: 0
- Missing parameters: 0
- Null and sentinel values: 0 across all 11 parameters
- Derived-feature test: VPD, temperature range, base-10 GDD and 35 °C heat-stress days calculated successfully
- Demonstration result: 109 days reached at least 35 °C during 2025 at the tested coordinate
- Invalid-parameter handling: HTTP 422 with descriptive messages
- Selected community: `AG` — Agroclimatology
- Default climatology period: January 2001–December 2020
- Production requirement: caching, provenance, timeouts and missing-value validation